# T35 - 不同曲线投资顺序分析
## 第1章：数据准备

### 概述
本章介绍项目的数据准备工作，包括：n1. 配置加载与环境设置
2. 数据库连接测试
3. 数据目录结构说明
4. 依赖包检查与安装

In [ ]:
# 1. 导入必要的库
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os
import sys
import warnings

warnings.filterwarnings('ignore')

# 设置显示选项
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.max_colwidth', 50)

print("库导入成功！")
print(f"当前工作目录: {os.getcwd()}")
print(f"Python版本: {sys.version}")

In [ ]:
# 2. 检查并安装必要的依赖包
import subprocess

required_packages = {
    'pandas': 'pandas',
    'numpy': 'numpy',
    'matplotlib': 'matplotlib',
    'seaborn': 'seaborn',
    'sqlalchemy': 'sqlalchemy',
    'pymysql': 'pymysql'
}

print("检查依赖包...")
for package, pip_name in required_packages.items():
    try:
        __import__(package)
        print(f"  [OK] {package}")
    except ImportError:
        print(f"  [安装] {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name, "--quiet"])
        print(f"  [完成] {package} 已安装")

print("\n所有依赖包检查完成！")

In [ ]:
# 3. 加载配置文件
from config import (
    print_config,
    get_database_url,
    DB_HOST, DB_PORT, DB_NAME,
    START_DATE, END_DATE,
    BOND_TYPES, RATE_BONDS, FINANCE_BONDS, CREDIT_BONDS,
    RATE_BOND_TERMS, CREDIT_BOND_TERMS,
    RATINGS, RATING_MAPPING,
    YIELD_DOWN_CYCLES,
    INITIAL_AMOUNT,
    OUTPUT_DIR, YIELD_DATA_FILE, CYCLE_LABEL_FILE
)

# 打印配置信息
print_config()

In [ ]:
# 4. 测试数据库连接
import sqlalchemy
from sqlalchemy import create_engine

def test_database_connection():
    """测试数据库连接"""
    try:
        # 获取数据库URL
        db_url = get_database_url()
        
        # 创建引擎
        engine = create_engine(
            db_url,
            poolclass=sqlalchemy.pool.NullPool,
            pool_recycle=3600
        )
        
        # 测试连接
        with engine.connect() as conn:
            result = conn.execute(sqlalchemy.text("SELECT 1"))
            print("数据库连接成功！")
            print(f"主机: {DB_HOST}")
            print(f"端口: {DB_PORT}")
            print(f"数据库: {DB_NAME}")
            return engine
    except Exception as e:
        print(f"数据库连接失败: {e}")
        print("\n提示: 请确保已设置环境变量 DB_PASSWORD")
        return None

# 测试连接
db_engine = test_database_connection()

In [ ]:
# 5. 检查数据目录结构
def check_directory_structure():
    """检查并创建必要的数据目录"""
    print("检查数据目录结构...")
    print(f"输出目录: {OUTPUT_DIR}")
    
    # 检查目录是否存在
    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR)
        print(f"  创建目录: {OUTPUT_DIR}")
    else:
        print(f"  目录已存在: {OUTPUT_DIR}")
    
    # 列出目录内容
    files = os.listdir(OUTPUT_DIR)
    print(f"\n目录内容 ({len(files)} 个文件):")
    for f in sorted(files):
        file_path = os.path.join(OUTPUT_DIR, f)
        if os.path.isfile(file_path):
            size = os.path.getsize(file_path)
            print(f"  - {f} ({size:,} bytes)")
        else:
            print(f"  - {f}/ (目录)")

check_directory_structure()

In [ ]:
# 6. 查看债券类型配置
print("债券类型配置")
print("=" * 60)

print("\n1. 全部债券类型:")
for i, bond_type in enumerate(BOND_TYPES, 1):
    print(f"   {i:2d}. {bond_type}")

print("\n2. 利率债类型:")
for bond_type in RATE_BONDS:
    print(f"   - {bond_type}")

print("\n3. 金融债类型:")
for bond_type in FINANCE_BONDS:
    print(f"   - {bond_type}")

print("\n4. 信用债类型:")
for bond_type in CREDIT_BONDS:
    print(f"   - {bond_type}")

In [ ]:
# 7. 查看期限配置
print("期限配置")
print("=" * 60)

print("\n1. 利率债期限区间:")
for term in RATE_BOND_TERMS:
    print(f"   - {term}年")

print("\n2. 信用债期限区间:")
for term in CREDIT_BOND_TERMS:
    print(f"   - {term}年")

print("\n3. 评级等级:")
for rating in RATINGS:
    print(f"   - {rating}")

print("\n4. 资质等级映射:")
for level, ratings in RATING_MAPPING.items():
    print(f"   - {level}: {', '.join(ratings)}")

In [ ]:
# 8. 查看历史周期配置
print("历史收益率下行周期")
print("=" * 60)

cycles_df = pd.DataFrame(YIELD_DOWN_CYCLES)
print(cycles_df.to_string(index=False))

print("\n周期统计:")
print(f"  总周期数: {len(YIELD_DOWN_CYCLES)}")
print(f"  平均收益率变动: {cycles_df['yield_change'].mean():.4f}%")
print(f"  最大收益率变动: {cycles_df['yield_change'].min():.4f}% (周期{cycles_df.loc[cycles_df['yield_change'].idxmin(), 'cycle']})")
print(f"  最小收益率变动: {cycles_df['yield_change'].max():.4f}% (周期{cycles_df.loc[cycles_df['yield_change'].idxmax(), 'cycle']})")

In [ ]:
# 9. 设置中文字体
import matplotlib.pyplot as plt
from matplotlib.font_manager import FontProperties

def setup_chinese_font():
    """设置中文字体"""
    font_paths = [
        r"C:\Windows\Fonts\msyh.ttc",  # 微软雅黑
        r"C:\Windows\Fonts\simhei.ttf",  # 黑体
        r"C:\Windows\Fonts\simsun.ttc",  # 宋体
    ]
    
    for font_path in font_paths:
        try:
            font = FontProperties(fname=font_path)
            print(f"成功加载中文字体: {font_path}")
            plt.rcParams['axes.unicode_minus'] = False  # 解决负号显示问题
            return font
        except:
            continue
    
    print("警告: 无法加载中文字体，图表中的中文可能无法正常显示")
    return None

chinese_font = setup_chinese_font()

### 总结

本章完成了以下工作：
1. 导入了必要的Python库
2. 检查并安装了依赖包
3. 加载并展示了配置信息
4. 测试了数据库连接
5. 检查了数据目录结构
6. 设置了中文字体

**下一步**: 运行 `02-周期标注.ipynb` 生成收益率下行周期标注数据。